In [ ]:
import os
import re
import numpy as np
import pandas as pd
import mne

from scipy.stats import entropy, skew, kurtosis
from scipy.signal import welch
from mne.filter import filter_data

In [ ]:
!pip install antropy
import antropy as ant

In [ ]:
BANDS = {'delta':[0.5, 4.0],
        'theta': [4.0, 8.0],
        'alpha':[8.0, 13.0],
        'beta': [13.0, 30.0],
        'gamma': [30.0, 100]}
FS = 256

In [ ]:
def trad_FTs(signal, fs=FS):
    features = {}

    # --- 1. Đặc trưng Thống kê Cơ bản ---
    features['mean'] = np.mean(signal)
    features['std'] = np.std(signal)
    features['skewness'] = skew(signal)
    features['kurtosis'] = kurtosis(signal)

    # --- 2. Đặc trưng Tần số (Frequency Domain) ---
    freqs, psd = welch(signal, fs=fs, nperseg=min(len(signal), 256))

    features['mean_spectral_power'] = np.mean(psd)
    features['max_spectral_power'] = np.max(psd)
    features['freq_of_max_power'] = freqs[np.argmax(psd)]

    # --- 3. Đặc trưng Phi tuyến & Entropy ---
    
    # Zero Crossing Rate
    features['zero_crossing_rate'] = (
        np.sum(np.diff(np.sign(signal)) != 0) / len(signal)
    )

    # Shannon Entropy
    hist, _ = np.histogram(signal, bins=10, density=True)
    features['shannon_entropy'] = entropy(hist + 1e-12)
    
    # Sample Entropy (built-in từ antropy)
    features['sample_entropy'] = ant.sample_entropy(np.asarray(signal, dtype=np.float32))
    return features

In [ ]:
def trad_fts_nd(data):
    """
    Vectorized version: processes all channels at once.
    data: shape (n_channels, n_samples)
    Returns a single-row DataFrame with columns like feature_channel.
    """
    rows = {}
    for ch in range(data.shape[0]):
        fts = trad_FTs(data[ch])          # dict of {feature_name: value}
        for feat_name, val in fts.items():
            rows[f"{feat_name}_{ch}"] = val
    return rows                            # return dict, concat later

In [ ]:
input_path = '/kaggle/input/datasets/ngoclannguyen00/chb23-segment/CHB-MIT_Segment'
for path in sorted(os.listdir(input_path)):
    chb_path = os.path.join(input_path, path, 'testing')
    chb_idx = path[-5:]
    print(f'Processing {chb_idx}')
    epoch_output = os.path.join(output_base, chb_idx)

    os.makedirs(epoch_output, exist_ok=True)
    seizures = os.listdir(chb_path)
    records = []
    for sz in sorted(seizures):
        sz_path = os.path.join(chb_path, sz)
        if not os.path.isdir(sz_path):
            continue
        inter = 0
        for file in sorted(os.listdir(sz_path)):
            if file.endswith(".npy"):
                full_path = os.path.join(sz_path, file)
                epochs = np.load(full_path)  # shape (n_epochs, n_channels, n_samples)
                
                for band, (lowcut, highcut) in BANDS.items():
                    temp_idx = inter
                    for epoch in enumerate(epochs):
                        epoch = filter_data(epoch, sfreq=FS, l_freq=lowcut, h_freq=highcut, method='fir', fir_design='firwin', verbose=False)
                        row = trad_fts_nd(epoch)
                        row['Band'] = band
                        row['SeizureID'] = sz
                        row['Epoch'] = temp_idx
                        records.append(row)
                        print(f'  [OK] for {chb_idx} | {band} | seizure {sz} | Epoch {temp_idx}')
                        temp_idx += 1
                inter = temp_idx
    df = pd.DataFrame(records)              
    save_file_path = os.path.join(epoch_output, f"{chb_idx}.csv")
    df.to_csv(save_file_path, index=False)
    del df, records, epoch


In [ ]:
for path in sorted(os.listdir(input_path)):
    chb_path = os.path.join(input_path, path, 'Training')
    chb_idx = path[-5:]
    print(f'Processing {chb_idx}')
    labels = os.listdir(chb_path)
    
    epoch_output = os.path.join(output_base, chb_idx)
    os.makedirs(epoch_output, exist_ok=True)
    for label in sorted(labels):
        label_path = os.path.join(chb_path, label)
        if not os.path.isdir(label_path):
            continue
        inter_epoch_idx = 0
        records = []
        for file in sorted(os.listdir(label_path)):
            if file.endswith(".npy"):
                file_name = os.path.splitext(file)[0] 
                full_path = os.path.join(label_path, file)
                epochs = np.load(full_path)  # shape (n_epochs, n_channels, n_samples)
                
                for band, (lowcut, highcut) in BANDS.items():
                    temp_inter_epoch_idx = inter_epoch_idx
                    for epoch_idx, epoch in enumerate(epochs):
                        epoch = filter_data(epoch, sfreq=FS, l_freq=lowcut, h_freq=highcut, method='fir', fir_design='firwin', verbose=False)
                        row = trad_fts_nd(epoch)
                        row['Band'] = band
                        if label=='preictal':
                            row['SeizureID'] = file_name
                            row['Epoch'] = epoch_idx
                        else:
                            row['Epoch'] = temp_inter_epoch_idx
                            
                        row['Class'] = label
                        
                        records.append(row)
                        if label=='preictal':
                            print(f'  [OK] for {chb_idx} sz {file_name} | {band} | {label} | Epoch {epoch_idx}')
                        else:
                            print(f'  [OK] for {chb_idx} | {band} | {label} | Epoch {temp_inter_epoch_idx}')
                            temp_inter_epoch_idx += 1
                inter_epoch_idx = temp_inter_epoch_idx
        df = pd.DataFrame(records)              
        save_file_path = os.path.join(epoch_output, f"{label}.csv")
        df.to_csv(save_file_path, index=False)
        del df, records, epoch
